# Nicha's song-gpt

### reading and exploring the data

- download ur datasets
- open input.txt and read all text as a string
- check if the first 1000 characters are correct
- chars = sorted(list(set(text))) → create a sorted list of chars
- num(chars) → vocabulary n

In [9]:
import os

all_text = ""

lyrics = "./data/data/Albums"

for root, dir, files in os.walk(lyrics):
        for file in files:
            if file.endswith(".txt"):
                with open(os.path.join(root, file), "r", encoding="utf-8") as f:
                    all_text += f.read()
                    all_text += "\n\n"

# write lyrics into combined folder
with open("input.txt", "w", encoding="utf-8") as f:
    f.write(all_text)

# read the combined text file and print the first 1000 characters
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text[:1000])

125 ContributorsTranslationsEspañolPortuguêsItalianoDeutschFrançaisTürkçeСрпски中文MagyarPolskiУкраїнська​evermore Lyrics[Verse 1: Taylor Swift]
Gray November
I've been down since July
Motion capture
Put me in a bad light
I replay my footsteps on each stepping stone
Trying to find the one where I went wrong
Writing letters
Addressed to the fire

[Chorus: Taylor Swift]
And I was catching my breath
Staring out an open window
Catching my death
And I couldn't be sure
I had a feeling so peculiar
That this pain would be for
Evermore
[Verse 2: Taylor Swift]
Hey December
Guess I'm feeling unmoored
Can't remember
What I used to fight for
I rewind thе tape, but all it does is pause
On thе very moment all was lost
Sending signals
To be double-crossed

[Chorus: Taylor Swift & Justin Vernon]
And I was catching my breath
Barefoot in the wildest winter
Catching my death
And I couldn't be sure
I had a feeling so peculiar
That this pain would be for
Evermore
(Evermore)
[Bridge: Justin Vernon, Taylor Swif

In [10]:
chars = sorted(list(set(text))) # vocab list -> all the possible letters the model can predict
vocab_size = len(chars)

print("vocab size:", vocab_size)

vocab size: 202


### tokenisation, train/val split

- tokenise = convert raw text as a string to some sequence of integers
- create a mapping from characters to integers → for encoder decoder
- encoder: take a string, output a list of integers
- decoder: take a list of integers, output a string
- gpt uses tiktoken
- encode entire text dataset and store it into a torch.Tensor
- separate into train/val split

In [12]:
import torch
import numpy

# char <-> int mapping 
char_to_int = {char:int for int, char in enumerate(chars)}
int_to_char = {int:char for int, char in enumerate(chars)}

# encode the text into integers
def encode(text):
    return [char_to_int[char] for char in text]

# decode the integers back into text
def decode(text):
    return "".join([int_to_char[int] for int in text])

encoded_text = encode(text)
data = torch.tensor(encoded_text, dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([2114880]) torch.int64
tensor([ 17,  18,  21,   2,  31,  72,  71,  77,  75,  66,  59,  78,  77,  72,
         75,  76,  48,  75,  58,  71,  76,  69,  58,  77,  66,  72,  71,  76,
         33,  76,  73,  58, 100,  72,  69,  44,  72,  75,  77,  78,  64,  78,
         97,  76,  37,  77,  58,  69,  66,  58,  71,  72,  32,  62,  78,  77,
         76,  60,  65,  34,  75,  58,  71,  95,  58,  66,  76,  48, 103,  75,
         68,  95,  62, 125, 138, 137, 139, 133, 131, 192, 193,  41,  58,  64,
         82,  58,  75,  44,  72,  69,  76,  68,  66, 126, 133, 138, 127, 142,
        135, 139, 141, 133, 127, 180,  62,  79,  62,  75,  70,  72,  75,  62,
          2,  40,  82,  75,  66,  60,  76,  55,  50,  62,  75,  76,  62,   2,
         17,  26,   2,  48,  58,  82,  69,  72,  75,   2,  47,  80,  66,  63,
         77,  56,   1,  35,  75,  58,  82,   2,  42,  72,  79,  62,  70,  59,
         62,  75,   1,  37,   8,  79,  62,   2,  59,  62,  62,  71,   2,  61,
         72,  80,  71,   2,  7

In [14]:
# train/val split
n = int(0.9*len(data))
train_data = data[:n] # first 90% of data for training
val_data = data[n:] # last 10% of data for validation

### data loader: batches of chunks of data

- we do not feed the entire text into the transformer all at once → computationally very expensive and prohibitive
    - we work with chunks of data at a time = block size (max length)
    - eg: tensor([18, 47, 56, 57, 58, 1, 15, 47, 58])
    - when input is tensor([18]) the target: 47
    - when input is tensor([18, 47]) the target: 56 etc
    - you want the transformer to see everything

In [16]:
torch.manual_seed(1337)
block_size = 8 # size of chunk model looks at
batch_size = 4 # num of chunks to process in parallel
n_embd = 32 # dimensionality of character embedding


# (4,) = vector with 4 elements 
def get_batch(split):
    data = train_data if split == "train" else val_data
    starting_pos = torch.randint(len(data) - block_size, (batch_size,)) 
    x = torch.stack([data[pos:pos+block_size] for pos in starting_pos])
    y = torch.stack([data[pos+1:pos+block_size+1] for pos in starting_pos])
    return x, y

xb, yb = get_batch("train")

### simplest baseline: bigram language model, loss, generation

- bigram model is the simplest possible language model — predicts the next character based on only the current character, no memory, no context just: given i see e, what usually comes next?
- a token is a character that is mapped to a number then after that u give each number n_embd # of numbers -> embedding layer
- forward() means: how data flows through the neural network, defines input → computation → output

In [ ]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super()._init()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    
    def forward(self, input, targets=None):
        logits = self.token_embedding_table(input) # (batch_size, block_size, n_embd)

        if targets is None:
            loss = None
        else:
            batch_size, block_size, n_embd = logits.shape
            logits = logits.view(batch_size*block_size, n_embd)
            targets = targets.view(batch_size*block_size) # flatten to (batch_size*block_size,)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss